# Module 2 — Deploy to AgentCore Runtime

In Module 1 your Chief of Staff agent ran on your laptop. Great for building — but no one else can reach
it. In this module you take the **exact same agent** and deploy it to **Amazon Bedrock AgentCore
Runtime**: a managed, serverless runtime that runs your agent in an isolated microVM and exposes it over
HTTP.

You'll deploy it **bare** — just the agent, in production. **No memory, no observability dashboards yet.**
That's deliberate: by the end you'll hit a real limitation (the agent forgets you between calls) that
**Module 3** fixes.

### What you'll do

| Step | What happens |
|------|--------------|
| **1. Recap the reuse** | See how the deployed agent reuses Module 1's `build_agent_options()` — one source of truth |
| **2. The entrypoint** | Understand the thin `@app.entrypoint` wrapper |
| **3. Configure** | Point AgentCore at your AWS account; review `agentcore.json` |
| **4. Test locally** | `agentcore dev` runs the container locally |
| **5. Deploy** | `agentcore deploy` builds the image, pushes to ECR, creates the runtime |
| **6. Invoke** | Call your live agent over HTTP and see the response |
| **7. Hit the wall** | Notice it's stateless — the hook into Module 3 |

## Why AgentCore Runtime?

Getting an agent to production normally means building session isolation, scaling, credential management,
and an HTTP layer yourself. AgentCore Runtime gives you all of that with a single deploy:

| Feature | What you get |
|---|---|
| **Isolated microVM per session** | Each session runs in its own sandbox |
| **Serverless & auto-scaling** | No servers to manage; scales with demand |
| **Managed identity/credentials** | The runtime gets a scoped IAM role automatically |
| **HTTP protocol** | Invoke over HTTP; streaming supported |

You bring the agent; AgentCore runs it.

## Setup

Run the cell below to install all dependencies and register the Jupyter kernel.
After it completes, **select the `module-2-deploy` kernel** from the kernel picker (top-right)
and continue with the rest of the notebook.

## Setup step 1

When you run the first script, it will ask you to select a environment

![](images/select-system-python.png)

and you can select the global env for now, and in this case it is 3.11.15 but this version may change

![](images/select-python-global-env.png)

once selected, you can rerun the setup.sh script

In [1]:
!bash setup.sh

2026-06-11 10:27:51 - Cleaning up old repositories...
2026-06-11 10:27:51 - Old repositories removed
2026-06-11 10:27:51 - Supported architecture: aarch64
2026-06-11 10:27:51 - Added N|Solid repository for LTS version: 20.x
2026-06-11 10:27:51 - dnf available, updating...
Node.js Packages for Linux RPM based distros -   19 MB/s | 1.1 MB     00:00    
Metadata cache created.
N|Solid Packages for Linux RPM based distros -   21 MB/s | 1.3 MB     00:00    
Metadata cache created.
2026-06-11 10:27:52 - Repository is configured and updated.
2026-06-11 10:27:52 - You can use N|solid Runtime as a node.js alternative
2026-06-11 10:27:52 - To install N|solid Runtime, run: dnf install nsolid -y
2026-06-11 10:27:52 - Run 'dnf install nodejs -y' to complete the installation.
Last metadata expiration check: 0:00:01 ago on Thu 11 Jun 2026 10:27:52 AM UTC.
Dependencies resolved.
 Package    Arch        Version                     Repository             Size
Installing:
 nodejs     aarch64     2:20.20.

### Setup step 2

once you see the depencies and kernel spec module-xxx are installed per message from the last step, please refresh your brower (not refresh kernel but brower)

![](images/refresh-browser.png)

and once refreshed, click on the kernel selector button at the top-right corner of the notebook (it probably shows Python 3.11.15)

![](images/current-python.png)

it will show you the option to select another kernel and please click

![](images/select-another-kernel.png)

once click, you will see the option of select jupyter kernel and please click on "Jupyter Kernel"

![](images/select-jupyter-kernel.png)

once click, you can see our registered module-x kernel, and the example shows modul-1-x but ***please select accordingly depedning on which model you are working on, if it is other module 2, then select module-2-xx for example***

![](images/example-select-module-1-jupter-kernel.png)

once selected, you will see the following as your kernel , ***please select accordingly depedning on which model you are working on, if it is other module 2, select module-2-xx kernel***

![](images/example-module-1-jupyter-kernel-selected.png)


## Step 1 — One agent, two front doors (the reuse)

We are **not** rewriting the agent. Module 1 and Module 2 share **one source of truth** for the agent's
identity — `build_agent_options()` in `agent.py`:

- **Module 1 (local):** `send_query()` calls `build_agent_options()` and runs the agent in-process.
- **Module 2 (deploy):** `agent_agentcore.py` calls the *same* `build_agent_options()` inside an AgentCore
  entrypoint.

Run the cell below to see that the deployment entrypoint contains **no agent logic of its own** — it just
wraps and reuses.

In [1]:
import sys
sys.path.insert(0, "chief_of_staff_agent")

import inspect
from agent import build_agent_options
import agent_agentcore

# The entrypoint imports the shared identity builder...
src = inspect.getsource(agent_agentcore)
assert "from agent import build_agent_options" in src
assert "build_agent_options()" in src
# ...and does NOT redefine the system prompt or tool list.
assert "system_prompt" not in src.replace("build_agent_options", "")
print("✅ agent_agentcore.py reuses build_agent_options() — no duplicated agent logic")

opts = build_agent_options()
print("   tools:", opts.allowed_tools)
print("   setting_sources:", opts.setting_sources)

✅ agent_agentcore.py reuses build_agent_options() — no duplicated agent logic
   tools: ['Task', 'Read', 'Write', 'Edit', 'Bash', 'WebSearch']
   setting_sources: ['project']


## Step 2 — The AgentCore entrypoint

AgentCore runs your agent through a small **entrypoint**: a handler that receives a request payload and
streams a response. Here is the whole thing (`chief_of_staff_agent/agent_agentcore.py`):

```python
from bedrock_agentcore import BedrockAgentCoreApp
from claude_agent_sdk import ClaudeSDKClient
from agent import build_agent_options          # ← reuse Module 1's identity

app = BedrockAgentCoreApp()

@app.entrypoint
async def invoke(payload: dict):
    prompt = (payload or {}).get("prompt")
    options = build_agent_options()             # ← same config as local
    async with ClaudeSDKClient(options=options) as agent:
        await agent.query(prompt)
        async for msg in agent.receive_response():
            for block in getattr(msg, "content", []) or []:
                if getattr(block, "text", None):
                    yield block.text             # ← stream text back

if __name__ == "__main__":
    app.run()                                    # serves /invocations + /ping on :8080
```

`BedrockAgentCoreApp` implements the runtime's HTTP contract (`/invocations`, `/ping`) for you. The Module
1 agent logic moves inside this handler **unchanged**, because it's the same `build_agent_options()`.

## Why a Container build (not a zip)?

AgentCore supports two build types: **CodeZip** (Python zipped to S3) and **Container** (a Docker image).
We use **Container**, and here's the concrete reason:

> The Claude Agent SDK ships a ~218MB **native CLI binary**. Packaged as a zip, that binary arrives in the
> runtime **without its execute permission**, and the agent dies at startup with
> `Permission denied: .../claude_agent_sdk/_bundled/claude`.

A Container build `pip install`s the SDK **inside a Linux/ARM64 image**, so the binary has the right
architecture and permissions. The `Dockerfile` lives in `chief_of_staff_agent/`. (AgentCore Runtime
requires `linux/arm64` images.)

## Step 3 — Configure the deployment

### What the AgentCore CLI already scaffolded for you

We used `agentcore create` + `agentcore add agent` to pre-build the project structure. Here's what
lives under `agentcore/`:

```
agentcore/
├── agentcore.json              # Project manifest — what to deploy and how
├── aws-targets.example.json    # Template for your deployment target (account + region)
└── cdk/                        # CDK app that synthesizes CloudFormation (you don't edit this)
    ├── bin/cdk.ts              #   CDK entrypoint
    ├── lib/cdk-stack.ts        #   Stack definition (auto-generated by the CLI)
    └── package.json            #   CDK dependencies
```

The CLI also generated the **Dockerfile** inside `chief_of_staff_agent/` (the `codeLocation`) — it
`pip install`s your agent's dependencies and sets the entrypoint.

### What `agentcore.json` declares

| Field | Meaning |
|-------|---------|
| `name` | Project name (`cosdeploy`) — used as a prefix for the CloudFormation stack |
| `managedBy` | `CDK` — the CLI uses AWS CDK under the hood to provision resources |
| `runtimes[0].name` | Agent name (`cos`) — identifies this agent within the project |
| `runtimes[0].build` | `Container` — builds a Docker image (vs. `CodeZip`). Required for the Claude Agent SDK (see Step 2) |
| `runtimes[0].entrypoint` | `agent_agentcore.py` — the file with `@app.entrypoint` |
| `runtimes[0].codeLocation` | `chief_of_staff_agent/` — everything in this folder goes into the container |
| `runtimes[0].networkMode` | `PUBLIC` — the agent can reach the internet (needed for Bedrock API calls) |
| `runtimes[0].protocol` | `HTTP` — exposes `/invocations` and `/ping` endpoints |
| `runtimes[0].instrumentation.enableOtel` | Sends OpenTelemetry traces to CloudWatch (explored in Module 4) |
| `runtimes[0].envVars` | Environment variables injected at runtime (model IDs, Bedrock flag) |
| `memories`, `credentials`, … | Empty for now — Module 3 will populate `memories[]` |

### What YOU need to configure: the deployment target

The only piece that's account-specific is **where** to deploy — your AWS account ID and region.
This goes in `agentcore/aws-targets.json` (gitignored because it contains your account ID).

The cell below auto-detects both from your current AWS credentials and writes the file:

In [2]:
import json, os, subprocess

# --- Resolve region: single source of truth for the whole notebook ---
# Priority: AWS_REGION env (set by Workshop Studio) → AWS CLI config → fallback
_cli_region = subprocess.run(
    ["aws", "configure", "get", "region"], capture_output=True, text=True
).stdout.strip()
REGION = os.environ.get("AWS_REGION") or _cli_region or "us-west-2"

account_id = subprocess.run(
    ["aws", "sts", "get-caller-identity", "--query", "Account", "--output", "text"],
    capture_output=True, text=True,
).stdout.strip()

# Generate aws-targets.json from the resolved values
targets = [
    {
        "name": "default",
        "description": "Workshop deployment target (auto-generated).",
        "account": account_id,
        "region": REGION,
    }
]

with open("agentcore/aws-targets.json", "w") as f:
    json.dump(targets, f, indent=2)

print(f"✅ agentcore/aws-targets.json written:")
print(f"   account: {account_id}")
print(f"   region:  {REGION}")

✅ agentcore/aws-targets.json written:
   account: 928448067233
   region:  us-east-1


In [3]:

#validate the configurations
!agentcore validate


The AgentCore CLI collects aggregated, anonymous usage
analytics to help improve the tool.
To opt out:          agentcore config telemetry.enabled false
To audit:            agentcore config telemetry.audit true
To learn more:       agentcore telemetry --help

Valid


Let's look at what we're about to deploy — the runtime entry in `agentcore.json`:

In [4]:
import json
cfg = json.load(open("agentcore/agentcore.json"))
print(json.dumps(cfg["runtimes"][0], indent=2))
# Note: build=Container, protocol=HTTP, enableOtel=true (traces → CloudWatch, explored in Module 4).
# The execution IAM role (Bedrock invoke + CloudWatch Logs + X-Ray) is created automatically by the CDK.

{
  "name": "cos",
  "build": "Container",
  "entrypoint": "agent_agentcore.py",
  "codeLocation": "chief_of_staff_agent/",
  "runtimeVersion": "PYTHON_3_11",
  "networkMode": "PUBLIC",
  "protocol": "HTTP",
  "instrumentation": {
    "enableOtel": true
  },
  "envVars": [
    {
      "name": "CLAUDE_CODE_USE_BEDROCK",
      "value": "1"
    },
    {
      "name": "ANTHROPIC_MODEL",
      "value": "global.anthropic.claude-opus-4-6-v1"
    },
    {
      "name": "ANTHROPIC_SMALL_FAST_MODEL",
      "value": "global.anthropic.claude-haiku-4-5-20251001-v1:0"
    }
  ]
}


## Step 4 — (Optional) Test locally first (`agentcore dev`)

Before deploying to AWS you can optionally run the agent locally in a container that mimics the
runtime. This is the fast dev loop — but it's a **long-running server** that blocks the kernel, so
run it in a **terminal**, from the repo root `cd` into this module's folder
`foundations/build-an-ai-chief-of-staff/module-2-deploy` first (not in a notebook cell):

```bash
# Terminal 1: from foundations/build-an-ai-chief-of-staff/module-2-deploy
cd foundations/build-an-ai-chief-of-staff/module-2-deploy
agentcore dev --no-browser    # start the local server (keep it running)
```

Then, in a **second terminal** (also in that folder), invoke it:

```bash
# Terminal 2: invoke the local agent
agentcore dev "What is our current monthly burn rate?" -p 8081
```

You should see the Chief of Staff answer using the company data — exactly like Module 1, but now
running through the AgentCore HTTP contract.

> **Tip:** You can also hit the endpoint directly with `curl` — this is the same contract
> AgentCore Runtime uses in production:
> ```bash
> curl -X POST http://localhost:8081/invocations \
>   -H "Content-Type: application/json" \
>   -d '{"prompt": "What is our current monthly burn rate?"}'
> ```

If you'd rather skip straight to deploying, continue to Step 5 — the live deployment will validate
everything the local test would.

## Step 5 — Deploy to AgentCore Runtime

This builds the container image, pushes it to ECR, and provisions the runtime via CDK. It takes a few
minutes and creates real AWS resources. Run it in a **terminal**, from the repo root `cd` into this
module's folder `foundations/build-an-ai-chief-of-staff/module-2-deploy` first (so the CLI finds the
`agentcore/` config):

```bash
cd foundations/build-an-ai-chief-of-staff/module-2-deploy
agentcore deploy -y
```

When it finishes you'll get a runtime ARN. Check status anytime (from the same folder):

```bash
agentcore status
```

In [5]:

# this may take a while and please wait for the deployment complete
!agentcore deploy -y


✓ Load deployment target
⠋ Validate project...(node:228695) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
✓ Validate project
✓ Build CDK project...
⠋ Synthesize CloudFormation...(node:228695) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
✓ Synthesize CloudFormation
✓ Check bootstrap status...
✓ Check stack status...
✓ Deploy to AW

In [6]:
!agentcore status

(node:232315) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
AgentCore Status (target: default, us-east-1)

Agents
  cos: Deployed - Runtime: READY (arn:aws:bedrock-agentcore:us-east-1:9284480672
33:runtime/cosdeploy_cos-uml09847Om)
  URL: https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/arn%3Aaws%3Abe
drock-agentcore%3Aus-east-1%3A928448067233%3Aruntime%2Fcosdeploy_cos-uml09847Om/
invocations

Update available: 0.17.0 → 0.19.0
Run `npm install -g @aws/agentcore@latest` to update.


In [7]:
# You can also check status from the notebook once deployed:
import subprocess
out = subprocess.run(["agentcore", "status"], capture_output=True, text=True)
print(out.stdout or out.stderr)

AgentCore Status (target: default, us-east-1)

Agents
  cos: Deployed - Runtime: READY (arn:aws:bedrock-agentcore:us-east-1:9284480672
33:runtime/cosdeploy_cos-uml09847Om)
  URL: https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/arn%3Aaws%3Abe
drock-agentcore%3Aus-east-1%3A928448067233%3Aruntime%2Fcosdeploy_cos-uml09847Om/
invocations



## Step 6 — Invoke your deployed agent

The agent is now live and reachable over HTTP. Invoke it:

In [8]:
!agentcore invoke --stream "What is our current runway and cash position?"

(node:232381) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
Here's a summary of our current financial position:

## 💰 Cash & Runway Overview

| Metric | Value |
|--------|-------|
| **Cash in Bank** | $10.0M |
| **Monthly Gross Burn** | ~$525K (as of June 2024) |
| **Monthly Revenue** | ~$290K (June 2024) |
| **Net Burn** | ~$235K/month |
| **Gross Runway** | ~20 months |
| **Net Runway** | ~42 months (accounting for revenue offset) |

## 📈 Key Trends

- **Burn is rising modestly** — from $450K (Jan) → $525K (Jun) due to headcount growth (45 → 53 employees).
- **Net burn is *improving*** — dropped from $270K → $235K/month 

That response was produced by your agent **running in AWS**, using the same `CLAUDE.md` company context
and the `financial-analysis` skill you built in Module 1 — now served from managed infrastructure.

> **Observability note:** traces for this invocation are already flowing to CloudWatch (we turned on
> `enableOtel`). We'll *explore* them in **Module 4**.

## Step 7 — The limitation you'll hit (hook into Module 3)

Try invoking twice, where the second call depends on the first:

```bash
agentcore invoke "My name is Sarah and I'm the CEO."
agentcore invoke "What's my name?"
```

The agent **won't remember**. Each invocation is independent — a fresh, isolated session. That's what
"stateless" means, and it's the right default for an auto-scaling runtime. But real products need to
remember their users.

::: That's exactly what **Module 3 — Add AgentCore Memory** fixes. :::

In [9]:
!agentcore invoke "My name is Sarah and I'm the CEO."

⠋ Invoking agent...(node:232767) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
nvoking agent...Good morning, Sarah! Great to connect with you. As your Chief of Staff, I'm here to help you stay on top of everything at TechStart — whether that's preparing for board meetings, analyzing our financial position, evaluating hiring decisions, or working through strategic questions.

What can I help you with today?

Session: 220c215f-9921-4a2d-9083-b2753c944d5b
To resume: agentcore invoke --session-id 220c215f-9921-4a2d-9083-b2753c944d5b
Log: /workshop/foundations/build-an-ai-chief-of-staff/module-2-deploy/agentcore/.cli/logs/invok

In [10]:
!agentcore invoke "What's my name?"

⠋ Invoking agent...(node:232823) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
I don't have information about your name. The context I have access to describes TechStart Inc's company details, team structure, and financials, but it doesn't include your personal identity.

Could you tell me your name, or let me know how I can help you today?

Session: 36fa9c59-624c-4001-bf68-f5379224743b
To resume: agentcore invoke --session-id 36fa9c59-624c-4001-bf68-f5379224743b
Log: /workshop/foundations/build-an-ai-chief-of-staff/module-2-deploy/agentcore/.cli/logs/invoke/invoke-cos-20260611-103700.log


## Key takeaways

- AgentCore Runtime gives you isolated, auto-scaling, managed hosting with a single `agentcore deploy`.
- You deployed your **existing** agent by wrapping it in a thin entrypoint that **reuses
  `build_agent_options()`** — no duplicated agent logic.
- The Claude Agent SDK's bundled binary means **Container builds** (Linux/ARM64), not zip.
- The CDK auto-creates the **execution IAM role**; `enableOtel` already ships traces to CloudWatch.
- A bare deployment is **stateless** — which is exactly what **Module 3 (Memory)** addresses next.

## Next steps

Continue to **Module 3 — Add AgentCore Memory** to make your deployed agent remember users across calls.